# Part 4 — Core API Deep-Dive (Human Example)

*Last updated:* 2026-11-12

This notebook is a **hands-on tutorial** of the public IDTrack API using **human**.

**Learning objectives**
- Load a graph into `idtrack.API` and understand what it wraps.
- Convert single identifiers (time travel + optional external outputs).
- Convert batches and summarize outcomes (1→0 / 1→1 / 1→n).
- Request explanation payloads for audit trails.
- Learn advanced knobs (external bridging, the ranking of candidates, search budgets, assembly awareness).
- Learn introspection helpers (available databases, assemblies, releases, active ranges).

> **Prerequisite:** `03_initialization_graph.ipynb` (Part 3) is recommended so the graph loads from cache.

## 4.1 — The API Facade

`idtrack.API` is the user-facing entry point. It handles:
- organism resolution (species names, common names and taxon IDs of the supported organisms: human, mouse, pig, rhesus and
  cynomolgus macaque)
- building or loading a graph snapshot (`API.build`, `API.load`; the reproducible snapshot boundary)
- conversions: `convert(...)` and `convert_many(...)`, which return typed results (`idtrack.results.ConversionResult`)

In this notebook we build (or load) the **human** snapshot, then use it for the rest of the examples.

> **Expected result:** after the setup cell runs, `api` holds the graph and conversions become available.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import API

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

organism = idtrack.organisms.resolve_organism('human')  # offline; API.ensembl_organism also asks Ensembl's newest release
SNAPSHOT_RELEASE = 115  # pinned

api = API.build(organism, SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
print('Ready:', organism, 'snapshot', SNAPSHOT_RELEASE)


def show(result, strategy='best'):
    """Print the main fields of a conversion result and its candidates of one view ('best' or 'all')."""
    print(f'{result.query_id!r} read as {result.graph_id!r} (match: {result.match}) at release {result.from_release} '
          f'-> {result.final_database} at {result.to_release}: status={result.status}, tie={result.tie}, '
          f'decided_by={result.decided_by}')
    for c in result.view(strategy):
        print(f'  {c.gene} -> {c.target}  best={c.best} entry={c.entry_gene} bridges={c.bridges} score={c.score:.3f}')
    if result.alternative_locus_genes:
        print('  copies on alternative loci:', list(result.alternative_locus_genes))
    if result.reason:
        print('  reason:', API.reason_text(result))

Ready: homo_sapiens snapshot 115


## 4.2 — Single Identifier Conversion


In [2]:
show(api.convert('TP53', to_release=SNAPSHOT_RELEASE))

'TP53' read as 'TP53' (match: exact) at release 115 -> ensembl_gene at 115: status=converted, tie=False, decided_by=None
  ENSG00000141510.20 -> ENSG00000141510.20  best=True entry=ENSG00000141510.20 bridges=0 score=1.000
  copies on alternative loci: ['LRG_321.1']


If the status is `not_found`, the input could not be matched to any identifier of the graph. IDTrack already tries the
letter-case and separator variants (`tp53`, `HLA_A` for `HLA-A`) and says in `match` how the identifier matched; `graph_id` is the
identifier of the graph it was read as, and `from_release` the release at which it was read. Without `from_release`, an identifier
is read at the target release if it exists there, otherwise at the latest release at which it exists.

`alternative_locus_genes` lists genes the identifier also leads to that are copies on alternative loci, assembly patches or LRG
records (`LRG_321.1` for TP53): they are reported apart from the answer, which holds the gene of the reference sequence.

### 4.2.1 Example: time travel (convert to an older release)

Why this matters: published datasets often use older releases.


In [3]:
# Choose an older release to demonstrate time travel
older_release = SNAPSHOT_RELEASE - 10
show(api.convert('TP53', to_release=older_release))

'TP53' read as 'TP53' (match: exact) at release 105 -> ensembl_gene at 105: status=converted, tie=False, decided_by=None
  ENSG00000141510.18 -> ENSG00000141510.18  best=True entry=ENSG00000141510.18 bridges=0 score=1.000
  copies on alternative loci: ['LRG_321.1']


### 4.2.2 Convert into an external database (HGNC)

To convert into a specific external database, pass `final_database=...`.
Database names are the same names you see in your external YAML.


In [4]:
show(api.convert('ENSG00000141510', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol'))

'ENSG00000141510' read as 'ENSG00000141510' (match: exact) at release 115 -> HGNC Symbol at 115: status=converted, tie=False, decided_by=None
  ENSG00000141510.20 -> TP53  best=True entry=ENSG00000141510.20 bridges=0 score=1.000


### 4.2.3 How do I know which external databases are available?

Ask the API: `external_databases()` lists the databases of the graph, `final_databases(to_release)` the values that
`final_database` accepts at a target release.

In [5]:
print(api.external_databases()[:50])
print(api.final_databases(SNAPSHOT_RELEASE)[:10])

['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT', 'Vega gene', 'Vega_gene', 'miRBase', 'miRBase label', 'synonym_id::HGNC Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)', 'synonym_id::UniProtKB Gene Name']
['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene (formerly Entrezgene)']


### 4.2.4 Understanding the result

A result is an `idtrack.results.ConversionResult` (a frozen dataclass; its docstring lists every field):
- `query_id`: exactly what you typed
- `graph_id`: the identifier of the graph it was read as; `match`: how it matched (`exact`, `case`, `separator`, `version`,
  `synonym:...`, `version_newer_than_graph`)
- `from_release`: the release it was read at; `from_release_inferred`: chosen by IDTrack (no `from_release` given);
  `from_release_fallback`: the `from_release` you gave does not hold the identifier, so it was read where the graph holds it
- `candidates`: every target the search reached, ranked, the best marked (`candidate.best`); `targets('best')` and
  `targets('all')` are the target ids of the two views (`view('best')`, `view('all')`)
- `tie`: the best answer is shared by several genes or targets equal in all evidence; `decided_by`: the criterion that put the
  best answer before the runner-up
- `alternative_locus_genes`: copies on alternative loci and patches reported apart from the answer
- `status`: `converted`, `tie`, `no_target`, `not_found` (input didn’t match any identifier of the graph), `unknown_version`
  (`known_versions` lists the versions that exist), `no_conversion` (input matched, but no gene at the target release descends
  from it) or `error`; `reason` says why a result has no candidate
- `no_target`: genes were reached, but none has an id in the requested database (the targets are then their Ensembl ids);
  `no_target_genes`: genes without such an id when others have one

Important: there can be 0, 1 or many targets, because ambiguity is real and common. Every result is strict JSON with
`result.to_dict()`, and `ConversionResult.from_dict(...)` reads it back.

### 4.2.5 Versions, spelling variants and releases

A few identifiers that show how IDTrack reads what you give it:
- an Ensembl id with a version that never existed is not answered with another version's gene (`known_versions` names the ones that
  exist), except a version higher than all the graph holds of an id still alive at the snapshot, which may come from a newer
  release and is read as the latest version;
- a `from_release` at which the graph does not hold the identifier is set aside and reported (`from_release_fallback`), while a
  release that is not a release of the graph at all is an error that names the valid ones.

In [6]:
from idtrack.errors import InputError

examples = [
    ('ENSG00000141510.5', {}),                      # an old version: read where it exists (release 54), followed to 115
    ('ENSG00000141510.3', {}),                      # a version the graph does not hold (its first is .4): nothing
    ('ENSG00000141510.25', {}),                     # newer than the graph's latest version: read as the latest version
    ('hla_a', {}),                                  # letter case and separator variant of HLA-A
    ('NM_000546.9', {}),                            # a RefSeq accession with a version the graph does not hold
    ('TP53', {'from_release': 50, 'to_release': 50}),  # read and converted at an NCBI36 release
    ('ENSG00000141510.20', {'from_release': 100}),  # this version did not exist at 100: read where it exists, flagged
    ('ENSG00000141510', {'from_release': 40}),      # not a release of the graph: an input error names the valid ones
]
for query, kwargs in examples:
    try:
        r = api.convert(query, **kwargs)
    except InputError as error:
        print(f'{query!r} {kwargs} -> {type(error).__name__}: {error}')
        continue
    versions = r.known_versions
    known = f' known_versions: {len(versions)}, {versions[0]} to {versions[-1]}' if versions else ''
    print(f"{query!r} {kwargs} -> {list(r.targets())} | graph_id={r.graph_id!r} match={r.match!r} "
          f"from_release={r.from_release} fallback={r.from_release_fallback} status={r.status}{known}")

'ENSG00000141510.5' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.5' match='exact' from_release=54 fallback=False status=converted


'ENSG00000141510.3' {} -> [] | graph_id=None match=None from_release=None fallback=False status=unknown_version known_versions: 17, ENSG00000141510.4 to ENSG00000141510.20


'ENSG00000141510.25' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='version_newer_than_graph' from_release=115 fallback=False status=converted
'hla_a' {} -> ['ENSG00000206503.16'] | graph_id='HLA-A' match='separator' from_release=115 fallback=False status=converted
'NM_000546.9' {} -> ['ENSG00000141510.20'] | graph_id='NM_000546' match='version' from_release=115 fallback=False status=converted
'TP53' {'from_release': 50, 'to_release': 50} -> ['ENSG00000141510.5'] | graph_id='TP53' match='exact' from_release=50 fallback=False status=converted
'ENSG00000141510.20' {'from_release': 100} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='exact' from_release=115 fallback=True status=converted
'ENSG00000141510' {'from_release': 40} -> UnknownReleaseError: `from_release`=40 is not a release of this graph, which covers the Ensembl releases 48 to 115.


### 4.2.6 Ambiguity: the best answer and every candidate

One conversion returns every candidate the search reached, ranked; the best answer is marked:
- `result.view('best')`: the best answer: one target, or, when several genes or targets are equal in all evidence, all of them
  with `tie=True`
- `result.view('all')`: *all* targets IDTrack found, best first; each candidate carries its ranking record (final confidence,
  bridges, gap, support of the entry gene, score, being the query itself, steps)

Use `'all'` when you are doing QC or want to inspect ambiguous mappings.

In [7]:
result = api.convert('TP53', to_release=SNAPSHOT_RELEASE)
show(result, 'all')
# The ranking record of the best candidate
result.view('best')[0]

'TP53' read as 'TP53' (match: exact) at release 115 -> ensembl_gene at 115: status=converted, tie=False, decided_by=None
  ENSG00000141510.20 -> ENSG00000141510.20  best=True entry=ENSG00000141510.20 bridges=0 score=1.000
  copies on alternative loci: ['LRG_321.1']


Candidate(gene='ENSG00000141510.20', target='ENSG00000141510.20', best=True, entry_gene='ENSG00000141510.20', entry_support=3, own_version=False, final_confidence=0, bridges=0, bridge_gap=0, score=1.0, same_as_query=False, external_steps=1, history_steps=0, annotation=None, path=(), final_path=())

## 4.3 — Batch Conversion

Most workflows start from a list of identifiers (genes in a count matrix, markers, hits, etc.).
IDTrack provides:
- `convert_many(...)`: one result per identifier; an error of one identifier never stops the batch
- `API.classify(...)` and `API.classification_summary(...)` to summarize outcomes

In [8]:
genes = ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS', 'NOT_A_REAL_GENE']
results = api.convert_many(genes, to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')
for r in results:
    print(f'{r.query_id:<16} {r.status:<10} {list(r.targets())}')

TP53             converted  ['TP53']
BRCA1            converted  ['BRCA1']
BRCA2            converted  ['BRCA2']
BRAF             converted  ['BRAF']
KRAS             converted  ['KRAS']
NOT_A_REAL_GENE  not_found  []


In [9]:
summary = API.classify(results)  # the bins of the best view; API.classify(results, 'all') for every target
# Each bin is a list of results
{k: len(v) for k, v in summary.items()}

{'changed_only_1_to_n': 0,
 'changed_only_1_to_1': 0,
 'alternative_target_1_to_1': 0,
 'alternative_target_1_to_n': 0,
 'matching_1_to_0': 1,
 'matching_1_to_1': 5,
 'matching_1_to_n': 0,
 'input_identifiers': 6}

If you want a human-readable report, print the summary of the bins:

In [10]:
print(API.classification_summary(summary))


IDTrack conversion summary:
  Total processed: 6
  1→0: 1 (16.7%)
  1→1: 5 (83.3%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
    Rest: 5 (100.0%)
  1→n: 0 (0.0%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
  Statuses:
    converted: 5
    not_found: 1


## 4.4 — Explainability & Auditability

When you set `explain=True`, the result includes `path`, an `idtrack.results.ExplainedPath`: one graph per conversion with every
node (its kind, label, databases, the releases at which a gene is active and the assemblies that serve them, the releases at
which it lies outside the reference sequence) and every step once. A step (`idtrack.results.PathStep`) has its kind (`entry`
from the identifier to the gene it stands for, `history` for an event of Ensembl's id history, `self_loop`, `bridge_near` and
`bridge_far` for the two sides of a bridge through an id two genes carried, `final` from the gene to its id in the final
database, `identity` for a gene that is its own answer), the releases, Ensembl's mapping score, the splits and merges of a
history event, the databases and the assembly of a cross-reference, and the reason it was taken. Each candidate lists its
steps by index: `path` from the identifier to the gene, `final_path` from the gene to the target.

In [11]:
explained = api.convert('TP53', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol', explain=True)
best = explained.view('best')[0]
for index in best.path + best.final_path:
    step = explained.path.steps[index]
    print(f'{step.kind:<8} {step.source} -> {step.target} | release {step.release} | {", ".join(step.databases)}')

entry    TP53 -> ENSG00000141510.20 | release 115 | HGNC Symbol, NCBI gene (formerly Entrezgene), UniProtKB Gene Name
final    ENSG00000141510.20 -> TP53 | release 115 | HGNC Symbol, NCBI gene (formerly Entrezgene), UniProtKB Gene Name


In [12]:
# The nodes of the explanation
for node in explained.path.nodes:
    print(node.kind, node.label, node.active, node.databases[:3])

external TP53 () ('HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'UniProtKB Gene Name')
ensembl_gene ENSG00000141510.20 ((115, None),) ()


The explanation is intentionally detailed. For most users, `status`, `tie` and the targets are enough.

## 4.5 — Advanced Conversion Options

`api.convert(...)` checks the arguments and reads the string, then calls the engine, `api.engine.convert(...)`
(`idtrack.engine.Track.convert`).

Use the high-level API most of the time; it already takes `allow_bridges`, `budget_states` and `budget_seconds`. The engine is
strict where the API is forgiving: a `from_release` at which the identifier does not exist raises
`idtrack.errors.InactiveAtReleaseError` (naming the releases where it exists), and an unknown version raises
`idtrack.errors.UnknownVersionError`.

### 4.5.1 Best vs all

- `result.view('best')` is the best answer: one target, or the targets that are equal in all evidence, as a tie (`tie=True`), in
  a fixed order.
- `result.view('all')` is every target reached, best first (useful for ambiguity-aware pipelines); `result.decided_by` names the
  criterion that put the best answer before the runner-up.

### 4.5.2 Controlling external bridging

The search follows Ensembl's history first. Only when that leads to no gene at the target release does it pass through an external
identifier or a transcript that two genes carried (a bridge): at most two bridges, the fewest possible, in the order of time, never
through clone-based names or hyper-connected identifiers. `candidate.bridges` says how many bridges were used. Switch bridging off
with `allow_bridges=False`. The search is bounded: each state is visited once, and a call that needs more than its budget
(`idtrack.settings.SearchSettings`: 200,000 states and 30 s by default; `budget_states=` and `budget_seconds=` of a call) raises
`idtrack.errors.SearchBudgetExceededError`.

### 4.5.3 Hyperconnected nodes

Some external identifiers are carried by many genes at the same time (RFAM families, `U6`, `5S_rRNA`, snoRNA names): more than
the hyper-connectivity threshold of the graph (`idtrack.settings.GraphSettings`, 5) genes in one release. They name a kind of gene, not a gene, so they are never used as bridges
and are not counted as annotation of a gene. They can still be queried and returned.

### 4.5.4 Assembly-aware conversions

In IDTrack, genome assemblies are part of the graph. This is crucial when you integrate datasets that were annotated with different
references (for example, a GRCh37-based GTF and a GRCh38-based GTF).

A conversion uses, at every release, the assembly Ensembl maintained at that release (human: NCBI36 up to release 54, GRCh37 up to
75, GRCh38 from 76). The parallel GRCh37 of later releases is not walked; its ids are in the graph as ids to convert from. The final
conversion into a database is made on the assembly of the target release, so a conversion to release 70 returns the HGNC symbols of
GRCh37 at release 70.

Practical consequences:
- You can feed identifiers originating from older builds and harmonize them into one target space (a release of your snapshot).
- External databases are read for the assembly they belong to; an id is returned as its database wrote it at the target release.

Example (advanced):
```python
api.convert(
    'TP53',
    to_release=SNAPSHOT_RELEASE,
    final_database='HGNC Symbol',
    from_database='HGNC Symbol',  # read the identifier as an id of this database only
    explain=True,
)
```

In [13]:
# Advanced demo: inspect hyperconnected nodes and compare `allow_bridges` behavior.
# Safe: does not modify your cache; it only runs conversions.

# 1) Hyperconnected nodes: external ids carried by more than the threshold of genes in one release
g = api.graph
hc = g.hyperconnective_nodes
print('Threshold (genes at once):', g.graph_settings.hyperconnective_threshold)
print('Hyperconnected external nodes:', len(hc))
top = sorted(hc.items(), key=lambda kv: (-kv[1], kv[0]))[:10]
print('Top 10 by the number of genes they are linked to in one release:')
for node, genes in top:
    print(' ', genes, '-', node)

# 2) External bridging toggle (often matters when backbone history is disconnected)
# For many well-behaved genes, both calls will succeed; the point is the *option* exists.
for allow_bridges in (False, True):
    res = api.convert('TP53', to_release=SNAPSHOT_RELEASE, allow_bridges=allow_bridges)
    print(f'allow_bridges={allow_bridges!s:<5} ->', {c.gene: c.bridges for c in res.view('best')})

Threshold (genes at once): 5
Hyperconnected external nodes: 10875
Top 10 by the number of genes they are linked to in one release:
  1562 - RF00026
  1337 - U6
  1266 - RF00017
  1013 - RF00019
  955 - Y_RNA
  905 - Metazoa_SRP
  615 - RF00001
  510 - 5S_rRNA
  480 - RF01210
  479 - snoU13
allow_bridges=False -> {'ENSG00000141510.20': 0}
allow_bridges=True  -> {'ENSG00000141510.20': 0}


## 4.6 — Introspection & Discovery

These helpers answer practical questions like:
- *Which external databases are available in my current graph?*
- *Which genome assemblies are represented?*
- *What Ensembl release range does my snapshot cover?*
- *When was a given identifier active across releases?*

The next cell demonstrates the most useful introspection calls.


In [14]:
# Introspection demo

print('Assemblies in this graph:', sorted(a.assembly for a in api.describe().assemblies))

ext_dbs = api.external_databases()
print('External DBs enabled (count):', len(ext_dbs))
print('External DBs (first 25):', ext_dbs[:25])

forms = api.database_forms()
print()
print('External DB connection forms (sample):')
for name in ext_dbs[:10]:
    print(' ', name, '→', forms.get(name))

rels = api.releases()
print()
print('Ensembl releases in snapshot window:', (min(rels), max(rels)) if rels else None)
described = api.describe()
print('Releases of each assembly:', {a.assembly: (a.name, min(a.releases), max(a.releases)) for a in described.assemblies})

# Active ranges: when was an ID "alive" across releases?
# (Useful for provenance documentation.)
g = api.graph
example_gene = 'ENSG00000141510.20'  # TP53 at the snapshot release
print()
print('Versions of ENSG00000141510:', g.versions_of('ENSG00000141510'))
print('Active ranges of', example_gene, '(the assembly Ensembl maintained at each release):', g.active_ranges(example_gene))
print('Active ranges of', example_gene, '(any assembly):', g.get_active_ranges_of_id_ensembl_all_inclusive(example_gene))

# Names per database and release: which genes does a name name, and which names does a gene carry?
print()
for match in api.genes_named('TP53')[-3:]:
    print('genes_named:', match.database, match.gene, match.releases[0], '-', match.releases[-1])
print('names_of:', [(m.database, m.name, m.synonym) for m in api.names_of(example_gene, SNAPSHOT_RELEASE)][:6])

2026-11-18 08:44:24 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


Assemblies in this graph: [36, 37, 38]
External DBs enabled (count): 28
External DBs (first 25): ['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT', 'Vega gene', 'Vega_gene', 'miRBase', 'miRBase label']

External DB connection forms (sample):
  CCDS → transcript
  CCDS label → transcript
  Clone_based_ensembl_gene → gene
  Clone_based_vega_gene → gene
  EntrezGene → gene
  HGNC Symbol → gene
  Havana gene → gene
  Havana transcript → transcript
  Havana translation → translation
  NCBI gene (formerly Entrezgene) → gene

Ensembl releases in snapshot window: (48, 115)
Releases of each assembly: {38: ('GRCh38', 76, 115), 37: ('GRCh37',

## 4.7 — Practical advice (the kind that saves you a week)

1. Always record your **snapshot boundary** (release) in your analysis notes.
2. If you share results, share the **external YAML** too.
3. When mapping is ambiguous, do not hide it — decide how your pipeline should handle 1→n mappings.
4. For scRNA-seq harmonization, prefer stable namespaces (Ensembl IDs) before switching to symbols.

> **Tip:** If you need troubleshooting checklists and diagnostics helpers, see `07_advanced_topics.ipynb` (Part 7.3).
